In [2]:
import os
import requests
import json
import psycopg2
from dotenv import load_dotenv
from smolagents import CodeAgent, tool, InferenceClientModel, LiteLLMModel, OpenAIModel
from urllib.parse import urlparse, urlunparse
from prompt import *
load_dotenv()
import litellm
import ast
litellm.set_debug = False

raw_url = os.getenv("SUPABASE_DB_URL")
parsed = urlparse(raw_url)
clean_url = urlunparse(parsed._replace(query=""))

from supabase import create_client
import os

url = os.getenv("url")
key = os.getenv("key")
supabase = create_client(url, key)

hf_token = os.getenv("HF_TOKEN")
supabase_auth = os.getenv("SUPABASE_DB_URL")

if not hf_token:
    print("ОШИБКА: HF_TOKEN не найден")
    exit(1)
if not supabase_auth:
    print("ОШИБКА: supabase_db_url не найден")
    exit(1)

def get_available_tests():
    try:
        response = supabase.rpc("run_query", {"sql": "SELECT distinct(test) from model_x_test"}).execute()
        if response.data:
            return [row['test'] for row in response.data]
        return []
    except:
        return ["Relative Performance TechPowerUp"]

available_tests = get_available_tests()

@tool
def query_database(sql: str) -> str:
    """
    Query the PC components database to get prices, benchmarks and compatibility info.
    Args:
        sql: SQL SELECT query to execute (no semicolons).
    """
    try:
        clean_sql = sql.strip().rstrip(";")
        response = supabase.rpc("run_query", {"sql": clean_sql}).execute()
        if response.data:
            return json.dumps(response.data, ensure_ascii=False, indent=2)
        return "[]"
    except Exception as e:
        return f"Database Error: {str(e)}"

tools = [
    query_database
]

def make_model(provider):
    if (provider=='hf'or provider=='huggingface'):
        model = InferenceClientModel(
        model_id="meta-llama/Llama-4-Scout-17B-16E-Instruct",
        token=hf_token
        )
        return model
    if (provider=='groq'):
        model = OpenAIModel(
        model_id="llama-3.3-70b-versatile",
        api_base="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY")
        )
        return model
    if (provider=='google'):
        model = OpenAIModel(
        model_id="gemini-2.0-flash", # или "gemini-1.5-flash"
        api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.getenv("GOOGLE_API_KEY")
        )
        return model

def make_agent(model, sys_prompt: str, max_steps: int = 8) -> CodeAgent:
    return CodeAgent(
        tools=[query_database],
        model=model,
        code_block_tags=("```python", "```"),
        additional_authorized_imports=['json', 're', 'pandas'],
        instructions=sys_prompt,
        step_callbacks=[lambda x: __import__('time').sleep(2.5)],
        max_steps=max_steps,
        verbosity_level=0
    )
model_ll = make_model('hf')
agent_init = make_agent(model_ll, prompts['init'])

query = "собери ПК за 120тр для киберпанк и кс2"

try:
    res = agent_init.run(query)
    if isinstance(res, str):
        clean_res = res.replace("", "").strip()
        try:
            request = ast.literal_eval(clean_res)
        except (ValueError, SyntaxError):
            request = res
    else:
        request = res
    print(request)
except Exception as e:
    print(f"\n❌ Ошибка исполнения: {e}")

if request['resolution'] == 0:
   if request['budget'] <= 110000: request['resolution'] = 1080
   if request['budget'] > 210000: request['resolution'] = 2160
   else: request['resolution'] = 1440

C:\Users\ya\Documents\agent_try\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ собери ПК за 120тр для киберпанк и кс2                                                                          │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-4-Scout-17B-16E-Instruct ──────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will extract the required parameters from the given Russian text. The text is: "собери ПК за 120тр для  
киберпанк и кс2".                                                                                                  
                                                                                                                   
- Budget: The budget is mentioned as "120тр", which is equivalent to 120,000 rubles.                               
- Mode: The request is for a full PC build since it mentions "ПК" (PC).                                            
- Task: The tasks mentioned are "киберпанк" (Cyberpunk) and "кс2" (CS2 or Counter-Strike 2). Since multiple tasks  
are mentioned, we will default to "Relative Performance TechPowerUp".                                              
- Resolution: No specific resolution is mentioned, so we will default to 0.                                        
                                                                                                                   
Let's construct the JSON object.                                                                                   
                                                                                                                   
```python                                                                                                          
budget = 120000                                                                                                    
mode = "assembly"                                                                                                  
task = "Relative Performance TechPowerUp"                                                                          
resolution = 0                                                                                                     
                                                                                                                   
json_output = "{\"budget\": " + str(budget) + ", \"mode\": \"" + mode + "\", \"task\": \"" + task + "\",           
\"resolution\": " + str(resolution) + "}"                                                                          
final_answer(json_output)                                                                                          
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  budget = 120000                                                                                                  
  mode = "assembly"                                                                                                
  task = "Relative Performance TechPowerUp"                                                                        
  resolution = 0                                                                                                   
                                                                                                                   
  json_output = "{\"budget\": " + str(budget) + ", \"mode\": \"" + mode + "\", \"task\": \"" + task + "\",         
  \"resolution\": " + str(resolution) + "}"                                                                        
  final_answer(json_output)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {"budget": 120000, "mode": "assembly", "task": "Relative Performance TechPowerUp", "resolution": 0}

[Step 1: Duration 9.97 seconds| Input tokens: 2,685 | Output tokens: 232]

{'budget': 120000, 'mode': 'assembly', 'task': 'Relative Performance TechPowerUp', 'resolution': 0}


In [3]:
import numpy as np

agent_gpu_ai = make_agent(model_ll, prompts['GPU_AI'])
agent_gpu = make_agent(model_ll, prompts['GPU'])

if request['mode'] == 'assembly':
    current_budget = request['budget']
    stages = ['GPU','CPU + MOTHERBOARD', 'RAM','PSU','STORAGE', 'CASE', 'COOLER']
    ratio = [[0.38, 0.51],
             [0.14, 0.22],
             [0.10, 0.16],
             [0.04, 0.08],
             [0.08, 0.16],
             [0.02, 0.06],
             [0.01, 0.04]]


In [5]:
min_price = request['budget']*ratio[0][0]
max_price = request['budget']*ratio[0][1]
target_task = request['task']
target_resolution = request['resolution']

if request['task'] == 'AI':
    temp = f"""
min_price = {min_price}, max_price={max_price}, target_task={target_task}, target_resolution={target_resolution}
"""
    gpu = dict(agent_gpu_ai.run(temp))
else:
    gpu_cnt_query = f"""
SELECT COUNT(*)
FROM gpus as g
INNER JOIN component_prices as p ON p.component_id = g.id
  AND p.is_available = TRUE AND p.is_verified = TRUE
  AND p.price_rub BETWEEN {min_price} AND {max_price}
INNER JOIN model_x_test as mxt
  ON mxt.model_id = g.model_id
  AND mxt.test = '{target_task}'
  AND mxt.resolution = '{target_resolution}'
"""
    response= supabase.rpc("run_query", {"sql": gpu_cnt_query}).execute()
    cnt = response.data[0].get('count') if response.data else 0
    if cnt==0:
        target_task='Relative Performance TechPowerUp'
    temp = f"""
min_price = {min_price}, max_price={max_price}, target_task={target_task}, target_resolution={target_resolution}
"""
try:
    if target_task == 'AI':
        raw_output = agent_gpu_ai.run(temp)
    else:
        raw_output = agent_gpu.run(temp)

    if isinstance(raw_output, str):
        clean_gpu = raw_output.replace("Final answer:", "").strip()
        gpu = ast.literal_eval(clean_gpu)
    else:
        gpu = raw_output

    print(gpu)
    print(f"Успешно получен тип: {type(gpu)}")

except Exception as e:
    print(f"\n❌ Ошибка исполнения: {e}")
    print(f"Сырой ответ от агента: {raw_output}")


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ min_price = 45600.0, max_price=61200.0, target_task=Relative Performance TechPowerUp, target_resolution=1440    │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-4-Scout-17B-16E-Instruct ──────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will start by querying the database to get the relevant GPU information within the given price range and
for the specified target task and resolution.                                                                      
                                                                                                                   
```python                                                                                                          
query = """                                                                                                        
SELECT *                                                                                                           
FROM (                                                                                                             
    SELECT                                                                                                         
        g.*,                                                                                                       
        p.price_rub,                                                                                               
        p.source_url,                                                                                              
        mxt.test,                                                                                                  
        mxt.result,                                                                                                
        MAX(mxt.result) OVER (PARTITION BY mxt.test) as max_result                                                 
    FROM gpus as g                                                                                                 
    INNER JOIN component_prices as p                                                                               
        ON p.component_id = g.id                                                                                   
        AND p.is_available = TRUE                                                                                  
        AND p.is_verified = TRUE                                                                                   
        AND p.price_rub BETWEEN 45600 AND 61200                                                                    
    INNER JOIN model_x_test as mxt                                                                                 
        ON mxt.model_id = g.model_id                                                                               
        AND mxt.test = 'Relative Performance TechPowerUp'                                                          
        AND mxt.resolution = 1440                                                                                  
) t                                                                                                                
WHERE result >= 0.85 * max_result                                                                                  
ORDER BY normalized_name, price_rub                                                                                
LIMIT 8000                                                                                                         
"""                                                                                                                
results = query_database(query)                                                                                    
print(results)                                                                                                     
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  query = """                                                                                                      
  SELECT *                                                                                                         
  FROM (                                                                                                           
      SELECT                                                                                                       
          g.*,                                                                                                     
          p.price_rub,                                                                                             
          p.source_url,                                                                                            
          mxt.test,                                                                                                
          mxt.result,                                                                                              
          MAX(mxt.result) OVER (PARTITION BY mxt.test) as max_result                                               
      FROM gpus as g                                                                                               
      INNER JOIN component_prices as p                                                                             
          ON p.component_id = g.id                                                                                 
          AND p.is_available = TRUE                                                                                
          AND p.is_verified = TRUE                                                                                 
          AND p.price_rub BETWEEN 45600 AND 61200                                                                  
      INNER JOIN model_x_test as mxt                                                                               
          ON mxt.model_id = g.model_id                                                                             
          AND mxt.test = 'Relative Performance TechPowerUp'                                                        
          AND mxt.resolution = 1440                                                                                
  ) t                                                                                                              
  WHERE result >= 0.85 * max_result                                                                                
  ORDER BY normalized_name, price_rub                                                                              
  LIMIT 8000                                                                                                       
  """                                                                                                              
  results = query_database(query)                                                                                  
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[
  {
    "id": "c3d83b5e-cb7c-42dc-b62c-4baf92fa235a",
    "tdp": 220,
    "test": "Relative Performance TechPowerUp",
    "result": 97.336,
    "model_id": "56478630-99fe-4ceb-863f-23841b5b218e",
    "length_mm": 288,
    "price_rub": 61138,
    "max_result": 97.336,
    "source_url": "https://www.wildberries.ru/catalog/542394046/detail.aspx",
    "normalized_name": "GIGABYTE AMD Radeon 9070 GAMING OC",
    "power_connectors": "16"
  },
  {
    "id": "1bf0b4bf-0fca-4bc6-a639-b314f301fab6",
    "tdp": 250,
    "test": "Relative Performance TechPowerUp",
    "result": 87.81,
    "model_id": "ee875a40-dc97-4e6d-95b3-c167c41a1007",
    "length_mm": 236,
    "price_rub": 58328,
    "max_result": 97.336,
    "source_url": "https://www.wildberries.ru/catalog/400753628/detail.aspx",
    "normalized_name": "MSI GeForce RTX 5070 VENTUS 2X OC",
    "power_connectors": "16"
  },
  {
    "id": "28619ba8-9c73-42c3-8fdc-9e37933fea14",
    "tdp": 250,
    "test": "Relative Performance TechPowerUp",
    "result": 87.81,
    "model_id": "ee875a40-dc97-4e6d-95b3-c167c41a1007",
    "length_mm": 332,
    "price_rub": 61138,
    "max_result": 97.336,
    "source_url": "https://www.wildberries.ru/catalog/760667535/detail.aspx",
    "normalized_name": "Palit GeForce RTX 5070 GamingPro",
    "power_connectors": "16"
  },
  {
    "id": "86d684c2-419b-49ad-8002-80f52b64b020",
    "tdp": 220,
    "test": "Relative Performance TechPowerUp",
    "result": 97.336,
    "model_id": "56478630-99fe-4ceb-863f-23841b5b218e",
    "length_mm": 280,
    "price_rub": 57544,
    "max_result": 97.336,
    "source_url": "https://www.wildberries.ru/catalog/834513944/detail.aspx",
    "normalized_name": "Sapphire AMD Radeon RX 9070 PULSE",
    "power_connectors": "16"
  }
]

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: We have obtained a list of GPUs that meet the specified criteria. Now, we need to apply the selection     
algorithm to choose the best GPU.                                                                                  
                                                                                                                   
Step 1: Since the target resolution is not 1080, we need to check if there are any GeForce GPUs in the results. If 
not, or if the Radeon GPU's result is better, we proceed with Radeon GPUs.                                         
                                                                                                                   
Step 2: From the filtered list, if any GPU has a TDP < 200, we pick the one with the lowest price. If all GPUs have
TDP >= 200, we pick the one with the lowest price and use length_mm as a tiebreaker.                               
                                                                                                                   
Let's analyze the results:                                                                                         
                                                                                                                   
- GIGABYTE AMD Radeon 9070 GAMING OC: TDP 220, Result 97.336, Price 61138                                          
- MSI GeForce RTX 5070 VENTUS 2X OC: TDP 250, Result 87.81, Price 58328                                            
- Palit GeForce RTX 5070 GamingPro: TDP 250, Result 87.81, Price 61138                                             
- Sapphire AMD Radeon RX 9070 PULSE: TDP 220, Result 97.336, Price 57544                                           
                                                                                                                   
All GPUs have TDP >= 200. We will choose the one with the lowest price. If there's a tie, we will use length_mm as 
a tiebreaker.                              

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import json                                                                                                      
                                                                                                                   
  # Results from the query                                                                                         
  gpus = [                                                                                                         
    {                                                                                                              
      "id": "c3d83b5e-cb7c-42dc-b62c-4baf92fa235a",                                                                
      "tdp": 220,                                                                                                  
      "test": "Relative Performance TechPowerUp",                                                                  
      "result": 97.336,                                                                                            
      "model_id": "56478630-99fe-4ceb-863f-23841b5b218e",                                                          
      "length_mm": 288,                                                                                            
      "price_rub": 61138,                                                                                          
      "max_result": 97.336,                                                                                        
      "source_url": "https://www.wildberries.ru/catalog/542394046/detail.aspx",                                    
      "normalized_name": "GIGABYTE AMD Radeon 9070 GAMING OC",                                                     
      "power_connectors": "16"                                                                                     
    },                                                                                                             
    {                                                                                                              
      "id": "1bf0b4bf-0fca-4bc6-a639-b314f301fab6",                                                                
      "tdp": 250,                                                                                                  
      "test": "Relative Performance TechPowerUp",                                                                  
      "result": 87.81,                                                                                             
      "model_id": "ee875a40-dc97-4e6d-95b3-c167c41a1007",                                                          
      "length_mm": 236,                                                                                            
      "price_rub": 58328,                                                                                          
      "max_result": 97.336,                                                                                        
      "source_url": "https://www.wildberries.ru/catalog/400753628/detail.aspx",                                    
      "normalized_name": "MSI GeForce RTX 5070 VENTUS 2X OC",                                                      
      "power_connectors": "16"                                                                                     
    },                                                                                                             
    {                                                                                                              
      "id": "28619ba8-9c73-42c3-8fdc-9e37933fea14",                                                                
      "tdp": 250,                                                                                                  
      "test": "Relative Performance TechPowerUp",        

Final answer: {"normalized_name": "Sapphire AMD Radeon RX 9070 PULSE", "price_rub": 57544, "tdp": 220, "length_mm":
280, "power_connectors": "16", "source_url": "https://www.wildberries.ru/catalog/834513944/detail.aspx"}

[Step 2: Duration 4.32 seconds| Input tokens: 6,936 | Output tokens: 1,356]

{'normalized_name': 'Sapphire AMD Radeon RX 9070 PULSE', 'price_rub': 57544, 'tdp': 220, 'length_mm': 280, 'power_connectors': '16', 'source_url': 'https://www.wildberries.ru/catalog/834513944/detail.aspx'}
Успешно получен тип: <class 'dict'>


In [6]:
agent_cpu_mb = make_agent(model_ll, prompts['CPU+MOTHERBOARD'])
min_price = request['budget']*ratio[1][0]
max_price = request['budget']*ratio[1][1]
if request['budget'] < 200000: ddr = 'DDR4'
else: ddr = 'DDR5'
temp = f"""
min_price = {min_price}, max_price={max_price}, target_task={target_task}, ram_type={ddr}
"""

In [8]:
try:
    raw_output = agent_cpu_mb.run(temp)
    if isinstance(raw_output, str):
        clean_str = raw_output.replace("Final answer:", "").strip()
        try:
            cpu_mb = ast.literal_eval(clean_str)
        except (ValueError, SyntaxError):
            cpu_mb = raw_output
    else:
        cpu_mb = raw_output

    if isinstance(cpu_mb, dict):
        print(cpu_mb)
    else:
        print(cpu_mb)

except Exception as e:
    print(f"\n❌ Ошибка: {e}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ min_price = 16800.0, max_price=26400.0, target_task=Relative Performance TechPowerUp, ram_type=DDR4             │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-4-Scout-17B-16E-Instruct ──────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will start by determining the type of task and then proceed to construct a SQL query to find the        
suitable CPU and motherboard based on the given parameters.                                                        
                                                                                                                   
Given parameters:                                                                                                  
- min_price = 16800.0                                                                                              
- max_price = 26400.0                                                                                              
- target_task = "Relative Performance TechPowerUp"                                                                 
- ram_type = "DDR4"                                                                                                
                                                                                                                   
The task type seems to be "Relative Performance TechPowerUp", which doesn't explicitly indicate gaming or          
professional. However, based on common practices, "Relative Performance TechPowerUp" might be closer to a gaming or
general performance task. Since it's not clearly specified, I will assume it's a gaming task for now.              
                                                                                                                   
```python                                                                                                          
min_price, max_price = 16800.0, 26400.0                                                                            
target_task = "Relative Performance TechPowerUp"                                                                   
ram_type = "DDR4"                                                                                                  
                                                                                                                   
# Since target_task is not clearly gaming or professional, let's assume it's gaming for simplicity.                
task_type = "gaming"                                                                                               
```                                                                                                                
                                                                                                                   
Next, I will construct the SQL query to fetch the required data.                                                   
                                                                                                                   
```python                                                                                                          
query = f"""                                                                                                       
WITH cpu_x_test_prices AS (                                                                                        
  SELECT                                                                                                           
    c.normalized_model_name,                                                                                       
    t.test, t.result,                                                                                              
    p.price_rub,                                                                                                   
    p.source_url,                                                                                                  
    c.tdp,                                                                                                         
    CASE                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  min_price, max_price = 16800.0, 26400.0                                                                          
  target_task = "Relative Performance TechPowerUp"                                                                 
  ram_type = "DDR4"                                                                                                
                                                                                                                   
  # Since target_task is not clearly gaming or professional, let's assume it's gaming for simplicity.              
  task_type = "gaming"                                                                                             
                                                                                                                   
  query = f"""                                                                                                     
  WITH cpu_x_test_prices AS (                                                                                      
    SELECT                                                                                                         
      c.normalized_model_name,                                                                                     
      t.test, t.result,                                                                                            
      p.price_rub,                                                                                                 
      p.source_url,                                                                                                
      c.tdp,                                                                                                       
      CASE                                                                                                         
          WHEN c.tdp < 70 THEN 'low'                                                                               
          WHEN c.tdp >= 70 AND c.tdp < 121 THEN 'mid'                                                              
          ELSE 'high'                                                                                              
      END AS tdp_tier,                                                                                             
      c.socket,                                                                                                    
      c.compatible_chipsets_no_bios_flash                                                                          
    FROM cpu_x_test AS t                                                                                           
    INNER JOIN cpus AS c ON c.id = t.cpu_id                                                                        
    INNER JOIN component_prices AS p                                                                               
        ON c.id = p.component_id                                                                                   
       AND p.is_available = TRUE                                                                                   
       AND p.is_verified = TRUE                                                                                    
  ),                                                                                                               
  motherboard_prices AS (                                                                                          
    SELECT                                                                                                         
      m.normalized_name,                                                                                           
      m.socket,                                                                                                    
      m.chipset,                                         

Execution logs:
[
  {
    "tdp": 65,
    "test": "Cinebench R23 Multi core",
    "result": 11500,
    "socket": "AM4",
    "cpu_url": "https://www.wildberries.ru/catalog/377609179/detail.aspx",
    "cpu_name": "AMD Ryzen 5 5600",
    "ram_type": "DDR4",
    "form_factor": "mATX",
    "num_ram_slots": 2,
    "cpu_power_pins": 4,
    "motherboard_url": "https://www.wildberries.ru/catalog/732500105/detail.aspx",
    "cpu_and_mb_price": 17228,
    "motherboard_name": "ASRock B550M-HDV",
    "required_cpu_power_pins": 4
  },
  {
    "tdp": 65,
    "test": "Cinebench R23 Multi core",
    "result": 11500,
    "socket": "AM4",
    "cpu_url": "https://www.wildberries.ru/catalog/377609179/detail.aspx",
    "cpu_name": "AMD Ryzen 5 5600",
    "ram_type": "DDR4",
    "form_factor": "mATX",
    "num_ram_slots": 4,
    "cpu_power_pins": 8,
    "motherboard_url": "https://www.wildberries.ru/catalog/283341464/detail.aspx",
    "cpu_and_mb_price": 18446,
    "motherboard_name": "MSI B550M PRO-VDH",
    "required_cpu_power_pins": 4
  },
  {
    "tdp": 65,
    "test": "Cinebench R23 Multi core",
    "result": 11500,
    "socket": "AM4",
    "cpu_url": "https://www.wildberries.ru/catalog/562016771/detail.aspx",
    "cpu_name": "AMD Ryzen 5 5600X",
    "ram_type": "DDR4",
    "form_factor": "mATX",
    "num_ram_slots": 2,
    "cpu_power_pins": 4,
    "motherboard_url": "https://www.wildberries.ru/catalog/732500105/detail.aspx",
    "cpu_and_mb_price": 18029,
    "motherboard_name": "ASRock B550M-HDV",
    "required_cpu_power_pins": 4
  },
  {
    "tdp": 65,
    "test": "Cinebench R23 Multi core",
    "result": 11500,
    "socket": "AM4",
    "cpu_url": "https://www.wildberries.ru/catalog/562016771/detail.aspx",
    "cpu_name": "AMD Ryzen 5 5600X",
    "ram_type": "DDR4",
    "form_factor": "mATX",
    "num_ram_slots": 2,
    "cpu_power_pins": 4,
    "motherboard_url": "https://www.wildberries.ru/catalog/732500107/detail.aspx",
    "cpu_and_mb_price": 17570,
    "motherboard_name": "ASRock B550M-HVS SE",
    "required_cpu_power_pins": 4
  },
  {
    "tdp": 65,
    "test": "Cinebench R23 Multi core",
    "result": 11500,
    "socket": "AM4",
    "cpu_url": "https://www.wildberries.ru/catalog/562016771/detail.aspx",
    "cpu_name": "AMD Ryzen 5 5600X",
    "ram_type": "DDR4",
    "form_factor": "mATX",
    "num_ram_slots": 4,
    "cpu_power_pins": 8,
    "motherboard_url": "https://www.wildberries.ru/catalog/172675047/detail.aspx",
    "cpu_and_mb_price": 17118,
    "motherboard_name": "Gigabyte B550M K",
    "required_cpu_power_pins": 4
  },
  {
    "tdp": 65,
    "test": "Cinebench R23 Multi core",
    "result": 11500,
    "socket": "AM4",
    "cpu_url": "https://www.wildberries.ru/catalog/562016771/detail.aspx",
    "cpu_name": "AMD Ryzen 5 5600X",
    "ram_type": "DDR4",
    "form_factor": "mATX",
    "num_ram_slots": 4,
    "cpu_power_pins": 8,
    "motherboard_url": "https://www.wildberries.ru/catalog/283341464/detail.aspx",
    "cpu_and_mb_price": 19247,
    "motherboard_name": "MSI B550M PRO-VDH",
    "required_cpu_power_pins": 4
  },
  {
    "tdp": 65,
    "test": "Cinebench R23 Multi core",
    "result": 13800,
    "socket": "AM4",
    "cpu_url": "https://www.wildberries.ru/catalog/363174828/detail.aspx",
    "cpu_name": "AMD Ryzen 7 5700X",
    "ram_type": "DDR4",
    "form_factor": "mATX",
    "num_ram_slots": 2,
    "cpu_power_pins": 4,
    "motherboard_url": "https://www.wildberries.ru/catalog/223427674/detail.aspx",
    "cpu_and_mb_price": 18075,
    "motherboard_name": "ASRock A520M-HVS",
    "required_cpu_power_pins": 4
  },
  {
    "tdp": 65,
    "test": "Cinebench R23 Multi core",
    "result": 13800,
    "socket": "AM4",
    "cpu_url": "https://www.wildberries.ru/catalog/363174828/detail.aspx",
    "cpu_name": "AMD Ryzen 7 5700X",
    "ram_type": "DDR4",
    "form_factor": "mATX",
    "num_ram_slots": 2,
    "cpu_power_pins": 4,
    "motherboard_url": "https://www.wildberries.ru/catalog/732500105/detail.aspx",
    "cpu_and_mb_pric

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: The error indicates that the `gaming_results` list is empty, meaning no results match the filter criteria.
This could be due to the strict filtering conditions. Let's relax the conditions slightly and also ensure that we  
handle the case where no results are found.                                                                        
                                                                                                                   
Given that the task type is "Relative Performance TechPowerUp", let's assume it's closer to a gaming task and      
prioritize "Cinebench R23 Single Core" test results.                                                               
                                                                                                                   
```python                                                                                                          
# Load results from observation                                                                                    
results = [                                                                                                        
  {                                                                                                                
    "tdp": 65,                                                                                                     
    "test": "Cinebench R23 Multi core",                                                                            
    "result": 11500,                                                                                               
    "socket": "AM4",                                                                                               
    "cpu_url": "https://www.wildberries.ru/catalog/377609179/detail.aspx",                                         
    "cpu_name": "AMD Ryzen 5 5600",                                                                                
    "ram_type": "DDR4",                                                                                            
    "form_factor": "mATX",                                                                                         
    "num_ram_slots": 2,                                                                                            
    "cpu_power_pins": 4,                                                                                           
    "motherboard_url": "https://www.wildberries.ru/catalog/732500105/detail.aspx",                                 
    "cpu_and_mb_price": 17228,                                                                                     
    "motherboard_name": "ASRock B550M-HDV",                                                                        
    "required_cpu_power_pins": 4                                                                                   
  },                                                                                                               
  # ... other results ...                                                                                          
]                                                                                                                  
                                                                                                                   
# Filter results for gaming (Cinebench R23 Single core) and within budget                                          
gaming_results = [result for result in results if result['test'] == 'Cinebench R23 Single core' and                
result['[38;2;165;214;

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Load results from observation                                                                                  
  results = [                                                                                                      
    {                                                                                                              
      "tdp": 65,                                                                                                   
      "test": "Cinebench R23 Multi core",                                                                          
      "result": 11500,                                                                                             
      "socket": "AM4",                                                                                             
      "cpu_url": "https://www.wildberries.ru/catalog/377609179/detail.aspx",                                       
      "cpu_name": "AMD Ryzen 5 5600",                                                                              
      "ram_type": "DDR4",                                                                                          
      "form_factor": "mATX",                                                                                       
      "num_ram_slots": 2,                                                                                          
      "cpu_power_pins": 4,                                                                                         
      "motherboard_url": "https://www.wildberries.ru/catalog/732500105/detail.aspx",                               
      "cpu_and_mb_price": 17228,                                                                                   
      "motherboard_name": "ASRock B550M-HDV",                                                                      
      "required_cpu_power_pins": 4                                                                                 
    },                                                                                                             
    # ... other results ...                                                                                        
  ]                                                                                                                
                                                                                                                   
  # Filter results for gaming (Cinebench R23 Single core) and within budget                                        
  gaming_results = [result for result in results if result['test'] == 'Cinebench R23 Single core' and              
  result['cpu_and_mb_price'] >= min_price and result['cpu_and_mb_price'] <= max_price]                             
                                                                                                                   
  # If no results found, try to find results without strict filtering                                              
  if not gaming_results:                                                                                           
      gaming_results = [result for result in results if result['cpu_and_mb_price'] >= min_price and                
  result['cpu_and_mb_price'] <= max_price]                                                                         
                                                                                                                   
  # Find the best result                                                                                           
  if gaming_results:                                                                                               
      best_result = max(gaming_results, key=lambda x: x['result'])                                                 
      final_answer({                                     

Final answer: {'cpu_name': 'AMD Ryzen 5 5600', 'motherboard_name': 'ASRock B550M-HDV', 'cpu_and_mb_price': 17228, 
'tdp': 65, 'socket': 'AM4', 'form_factor': 'mATX', 'ram_type': 'DDR4', 'num_ram_slots': 2, 'cpu_power_pins': 4, 
'required_cpu_power_pins': 4, 'motherboard_url': 'https://www.wildberries.ru/catalog/732500105/detail.aspx', 
'cpu_url': 'https://www.wildberries.ru/catalog/377609179/detail.aspx'}

[Step 3: Duration 3.93 seconds| Input tokens: 28,505 | Output tokens: 2,485]

{'cpu_name': 'AMD Ryzen 5 5600', 'motherboard_name': 'ASRock B550M-HDV', 'cpu_and_mb_price': 17228, 'tdp': 65, 'socket': 'AM4', 'form_factor': 'mATX', 'ram_type': 'DDR4', 'num_ram_slots': 2, 'cpu_power_pins': 4, 'required_cpu_power_pins': 4, 'motherboard_url': 'https://www.wildberries.ru/catalog/732500105/detail.aspx', 'cpu_url': 'https://www.wildberries.ru/catalog/377609179/detail.aspx'}


In [9]:
agent_ram = make_agent(model_ll, prompts['RAM'])
max_price = request['budget'] * ratio[2][1]

temp = f"""
max_price={max_price}, ram_type={ddr}, target_task={query}
"""
try:
    res = agent_ram.run(temp)
    if isinstance(res, str):
        clean_res = res.replace("Final answer:", "").strip()
        try:
            ram = ast.literal_eval(clean_res)
        except Exception:
            ram = res
    else:
        ram = res
    print(ram)
except Exception as e:
    print(f"\n❌ Ошибка исполнения: {e}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ max_price=19200.0, ram_type=DDR4, target_task=собери ПК за 120тр для киберпанк и кс2                            │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-4-Scout-17B-16E-Instruct ──────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to solve this task by querying the database with the given parameters: `max_price=19200.0`,        
`ram_type=DDR4`, and `target_task=собери ПК за 120тр для киберпанк и кс2`. The task seems to be for gaming, so I   
will filter and sort the results accordingly.                                                                      
                                                                                                                   
First, I will translate the target task to understand it better. It seems like the task is to assemble a PC for 120
thousand rubles for playing Cyberpunk and CS2.                                                                     
                                                                                                                   
Given that the task is for gaming, I will:                                                                         
1. Filter for `total_capacity_gb` >= 16.                                                                           
2. Sort by `speed_mhz` DESC, then `total_capacity_gb` DESC, then `price_rub` ASC.                                  
                                                                                                                   
I will now construct the SQL query and execute it.                                                                 
                                                                                                                   
```python                                                                                                          
sql_query = """                                                                                                    
SELECT r.normalized_name, r.total_capacity_gb, r.number_of_modules, r.speed_mhz, p.price_rub, p.source_url         
FROM ram AS r                                                                                                      
INNER JOIN component_prices AS p ON r.id = p.component_id                                                          
WHERE r.type = 'DDR4'                                                                                              
  AND p.price_rub <= 19200.0                                                                                       
  AND r.number_of_modules = 2                                                                                      
  AND p.is_available = TRUE                                                                                        
UNION ALL                                                                                                          
SELECT '2x ' || r.normalized_name, 2*r.total_capacity_gb, 2, r.speed_mhz, 2*p.price_rub, p.source_url              
FROM ram AS r                                                                                                      
INNER JOIN component_prices AS p ON r.id = p.component_id                                                          
WHERE r.type = 'DDR4'                                                                                              
  AND p.price_rub <= 19200.0 / 2                                                                                   
  AND r.number_of_modules = 1                                                                                      
  AND p.is_available = TRUE                                                                                        
"""                                                                                                                
                                                                                                                   
# Since the task is for gaming, filter and sort results                                                            
# Filter for total_capacity_gb >= 16                    

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  sql_query = """                                                                                                  
  SELECT r.normalized_name, r.total_capacity_gb, r.number_of_modules, r.speed_mhz, p.price_rub, p.source_url       
  FROM ram AS r                                                                                                    
  INNER JOIN component_prices AS p ON r.id = p.component_id                                                        
  WHERE r.type = 'DDR4'                                                                                            
    AND p.price_rub <= 19200.0                                                                                     
    AND r.number_of_modules = 2                                                                                    
    AND p.is_available = TRUE                                                                                      
  UNION ALL                                                                                                        
  SELECT '2x ' || r.normalized_name, 2*r.total_capacity_gb, 2, r.speed_mhz, 2*p.price_rub, p.source_url            
  FROM ram AS r                                                                                                    
  INNER JOIN component_prices AS p ON r.id = p.component_id                                                        
  WHERE r.type = 'DDR4'                                                                                            
    AND p.price_rub <= 19200.0 / 2                                                                                 
    AND r.number_of_modules = 1                                                                                    
    AND p.is_available = TRUE                                                                                      
  """                                                                                                              
                                                                                                                   
  # Since the task is for gaming, filter and sort results                                                          
  # Filter for total_capacity_gb >= 16                                                                             
  # Sort by speed_mhz DESC, then total_capacity_gb DESC, then price_rub ASC                                        
                                                                                                                   
  print(sql_query)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:

SELECT r.normalized_name, r.total_capacity_gb, r.number_of_modules, r.speed_mhz, p.price_rub, p.source_url
FROM ram AS r
INNER JOIN component_prices AS p ON r.id = p.component_id
WHERE r.type = 'DDR4'
  AND p.price_rub <= 19200.0
  AND r.number_of_modules = 2
  AND p.is_available = TRUE
UNION ALL
SELECT '2x ' || r.normalized_name, 2*r.total_capacity_gb, 2, r.speed_mhz, 2*p.price_rub, p.source_url
FROM ram AS r
INNER JOIN component_prices AS p ON r.id = p.component_id
WHERE r.type = 'DDR4'
  AND p.price_rub <= 19200.0 / 2
  AND r.number_of_modules = 1
  AND p.is_available = TRUE


Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: The results have been fetched successfully. Since the task is for gaming, I will now filter the results to
ensure `total_capacity_gb` >= 16 and then sort them by `speed_mhz` DESC, `total_capacity_gb` DESC, and `price_rub` 
ASC.                                                                                                               
                                                                                                                   
However, since there is only one result and it meets the criteria, I will directly construct the final JSON and    
provide it as the answer.                                                                                          
                                                                                                                   
```python                                                                                                          
result = {                                                                                                         
    "normalized_name": "Kingston Fury DDR4 3200MHz 16Gb",                                                          
    "total_capacity_gb": 16,                                                                                       
    "speed_mhz": 3200,                                                                                             
    "price_rub": 12700,                                                                                            
    "source_url": "https://www.wildberries.ru/catalog/150972149/detail.aspx"                                       
}                                                                                                                  
                                                                                                                   
final_answer(result)                                                                                               
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = {                                                                                                       
      "normalized_name": "Kingston Fury DDR4 3200MHz 16Gb",                                                        
      "total_capacity_gb": 16,                                                                                     
      "speed_mhz": 3200,                                                                                           
      "price_rub": 12700,                                                                                          
      "source_url": "https://www.wildberries.ru/catalog/150972149/detail.aspx"                                     
  }                                                                                                                
                                                                                                                   
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'normalized_name': 'Kingston Fury DDR4 3200MHz 16Gb', 'total_capacity_gb': 16, 'speed_mhz': 3200, 
'price_rub': 12700, 'source_url': 'https://www.wildberries.ru/catalog/150972149/detail.aspx'}

[Step 3: Duration 1.32 seconds| Input tokens: 10,369 | Output tokens: 842]

{'normalized_name': 'Kingston Fury DDR4 3200MHz 16Gb', 'total_capacity_gb': 16, 'speed_mhz': 3200, 'price_rub': 12700, 'source_url': 'https://www.wildberries.ru/catalog/150972149/detail.aspx'}


In [10]:
agent_psu = make_agent(model_ll, prompts['PSU'])
min_price = request['budget'] * ratio[3][0]
max_price = request['budget'] * ratio[3][1]
gpu_tdp=int(gpu['tdp'])
cpu_tdp=int(cpu_mb['tdp'])
selected_mb_pins = int(cpu_mb['cpu_power_pins'])
selected_gpu_pins = int(gpu['power_connectors'])


temp = f"""
min_price = {min_price}, max_price={max_price}, gpu_tdp={gpu_tdp}, cpu_tdp={cpu_tdp
}, selected_mb_pins = {selected_mb_pins}, selected_gpu_pins = {selected_gpu_pins}
"""
try:
    res = agent_psu.run(temp)
    if isinstance(res, str):
        clean_res = res.replace("Final answer:", "").strip()
        try:
            psu = ast.literal_eval(clean_res)
        except Exception:
            psu = res
    else:
        psu = res
    print(psu)
except Exception as e:
    print(f"\n❌ Ошибка исполнения: {e}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ min_price = 4800.0, max_price=9600.0, gpu_tdp=220, cpu_tdp=65, selected_mb_pins = 4, selected_gpu_pins = 16     │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-4-Scout-17B-16E-Instruct ──────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: First, I need to calculate the required wattage based on the provided formula: `required_wattage = 1.5 *  
(gpu_tdp + cpu_tdp) + 50`. Then, I will use this value in the SQL query to find suitable PSUs.                     
                                                                                                                   
```python                                                                                                          
# Input values                                                                                                     
min_price = 4800.0                                                                                                 
max_price = 9600.0                                                                                                 
gpu_tdp = 220                                                                                                      
cpu_tdp = 65                                                                                                       
selected_mb_pins = 4                                                                                               
selected_gpu_pins = 16                                                                                             
                                                                                                                   
# Calculate required wattage                                                                                       
required_wattage = 1.5 * (gpu_tdp + cpu_tdp) + 50                                                                  
print(required_wattage)                                                                                            
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Input values                                                                                                   
  min_price = 4800.0                                                                                               
  max_price = 9600.0                                                                                               
  gpu_tdp = 220                                                                                                    
  cpu_tdp = 65                                                                                                     
  selected_mb_pins = 4                                                                                             
  selected_gpu_pins = 16                                                                                           
                                                                                                                   
  # Calculate required wattage                                                                                     
  required_wattage = 1.5 * (gpu_tdp + cpu_tdp) + 50                                                                
  print(required_wattage)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
477.5

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: The error indicates that the `results` variable is not a list of dictionaries, but rather a list of       
strings. This is likely due to the JSON output not being properly parsed. I will first parse the JSON output and   
then apply the selection algorithm.                                                                                
                                                                                                                   
```python                                                                                                          
import json                                                                                                        
                                                                                                                   
# Parse JSON output                                                                                                
results = json.loads('''                                                                                           
[                                                                                                                  
  {                                                                                                                
    "wattage": 850,                                                                                                
    "price_rub": 5040,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/805452158/detail.aspx?targetUrl=MI",                         
    "certificate": "Gold",                                                                                         
    "form_factor": "SFX",                                                                                          
    "cpu_pins_total": 16,                                                                                          
    "gpu_pins_total": 16,                                                                                          
    "normalized_name": "THERMALRIGHT 850W Gold SFX 3.1"                                                            
  },                                                                                                               
  {                                                                                                                
    "wattage": 750,                                                                                                
    "price_rub": 6090,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/561307479/detail.aspx",                                      
    "certificate": "Gold",                                                                                         
    "form_factor": "ATX",                                                                                          
    "cpu_pins_total": 16,                                                                                          
    "gpu_pins_total": 16,                                                                                          
    "normalized_name": "1stPlayer NGDP 750W"                                                                       
  },                                                                                                               
  {                                                                                                                
    "wattage": 750,                                                                                                
    "price_rub": 6090,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/56

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import json                                                                                                      
                                                                                                                   
  # Parse JSON output                                                                                              
  results = json.loads('''                                                                                         
  [                                                                                                                
    {                                                                                                              
      "wattage": 850,                                                                                              
      "price_rub": 5040,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/805452158/detail.aspx?targetUrl=MI",                       
      "certificate": "Gold",                                                                                       
      "form_factor": "SFX",                                                                                        
      "cpu_pins_total": 16,                                                                                        
      "gpu_pins_total": 16,                                                                                        
      "normalized_name": "THERMALRIGHT 850W Gold SFX 3.1"                                                          
    },                                                                                                             
    {                                                                                                              
      "wattage": 750,                                                                                              
      "price_rub": 6090,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/561307479/detail.aspx",                                    
      "certificate": "Gold",                                                                                       
      "form_factor": "ATX",                                                                                        
      "cpu_pins_total": 16,                                                                                        
      "gpu_pins_total": 16,                                                                                        
      "normalized_name": "1stPlayer NGDP 750W"                                                                     
    },                                                                                                             
    {                                                                                                              
      "wattage": 750,                                                                                              
      "price_rub": 6090,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/561307479/detail.aspx",                                    
      "certificate": "Gold",                                                                                       
      "form_factor": "ATX",                                                                                        
      "cpu_pins_total": 16,                                                                                        
      "gpu_pins_total": 16,                                                                                        
      "normalized_name": "1stPlayer NGDP 750W"           

Final answer: {'normalized_name': 'Lian Li EDGE 1000W', 'wattage': 1000, 'efficiency_rating': 'Platinum', 
'form_factor': 'ATX', 'price_rub': 7021, 'source_url': 
'https://www.wildberries.ru/catalog/766636293/detail.aspx?targetUrl=MI'}

[Step 6: Duration 5.32 seconds| Input tokens: 25,141 | Output tokens: 2,447]

{'normalized_name': 'Lian Li EDGE 1000W', 'wattage': 1000, 'efficiency_rating': 'Platinum', 'form_factor': 'ATX', 'price_rub': 7021, 'source_url': 'https://www.wildberries.ru/catalog/766636293/detail.aspx?targetUrl=MI'}


In [11]:
agent_disk = make_agent(model_ll, prompts['STORAGE'])
min_price = request['budget'] * ratio[4][0]
max_price = request['budget'] * ratio[4][1]
if request['budget'] < 50000: min_capacity = 240
elif request['budget'] > 15000: min_capacity = 1000
else: min_capacity = 500

temp = f"""
min_price = {min_price}, max_price={max_price}, min_capacity_gb={min_capacity}
"""
try:
    res = agent_disk.run(temp)
    if isinstance(res, str):
        clean_res = res.replace("Final answer:", "").strip()
        try:
            disk = ast.literal_eval(clean_res)
        except Exception:
            disk = res
    else:
        disk = res
    print(disk)
except Exception as e:
    print(f"\n❌ Ошибка исполнения: {e}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ min_price = 9600.0, max_price=19200.0, min_capacity_gb=1000                                                     │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-4-Scout-17B-16E-Instruct ──────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will start by querying the database with the given parameters: min_price = 9600.0, max_price = 19200.0, 
and min_capacity_gb = 1000. I will use these values directly in the SQL query.                                     
                                                                                                                   
```python                                                                                                          
sql_query = """                                                                                                    
SELECT s.normalized_name, s.capacity_gb, s.speed_tier, s.reliability_tier, p.price_rub, p.source_url               
FROM storage AS s                                                                                                  
INNER JOIN component_prices AS p ON s.id = p.component_id                                                          
WHERE s.capacity_gb >= 1000                                                                                        
  AND p.price_rub BETWEEN 9600 AND 19200                                                                           
  AND p.is_available = TRUE                                                                                        
  AND p.is_verified = TRUE                                                                                         
ORDER BY p.price_rub ASC                                                                                           
"""                                                                                                                
                                                                                                                   
result = query_database(sql_query)                                                                                 
print(result)                                                                                                      
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  sql_query = """                                                                                                  
  SELECT s.normalized_name, s.capacity_gb, s.speed_tier, s.reliability_tier, p.price_rub, p.source_url             
  FROM storage AS s                                                                                                
  INNER JOIN component_prices AS p ON s.id = p.component_id                                                        
  WHERE s.capacity_gb >= 1000                                                                                      
    AND p.price_rub BETWEEN 9600 AND 19200                                                                         
    AND p.is_available = TRUE                                                                                      
    AND p.is_verified = TRUE                                                                                       
  ORDER BY p.price_rub ASC                                                                                         
  """                                                                                                              
                                                                                                                   
  result = query_database(sql_query)                                                                               
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[
  {
    "price_rub": 11807,
    "source_url": "https://www.wildberries.ru/catalog/292668087/detail.aspx",
    "speed_tier": "low",
    "capacity_gb": 2000,
    "normalized_name": "WD Purple 2Tb",
    "reliability_tier": "mid"
  },
  {
    "price_rub": 12663,
    "source_url": "https://www.wildberries.ru/catalog/174666786/detail.aspx",
    "speed_tier": "high",
    "capacity_gb": 1000,
    "normalized_name": "MSI SPATIUM M461",
    "reliability_tier": "high"
  },
  {
    "price_rub": 13453,
    "source_url": "https://www.wildberries.ru/catalog/178970443/detail.aspx",
    "speed_tier": "low",
    "capacity_gb": 4000,
    "normalized_name": "Seagate Barracuda 4Tb",
    "reliability_tier": "mid"
  },
  {
    "price_rub": 16904,
    "source_url": "https://www.wildberries.ru/catalog/194967364/detail.aspx",
    "speed_tier": "low",
    "capacity_gb": 6000,
    "normalized_name": "Seagate Barracuda 6Tb",
    "reliability_tier": "mid"
  },
  {
    "price_rub": 17335,
    "source_url": "https://www.wildberries.ru/catalog/792277180/detail.aspx",
    "speed_tier": "high",
    "capacity_gb": 1000,
    "normalized_name": "Samsung 9100 Pro 1Tb",
    "reliability_tier": "high"
  }
]

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: The error occurred because the `result` variable is a string, not a list of dictionaries. I will first    
convert the string into a JSON object using the `json` module.                                                     
                                                                                                                   
```python                                                                                                          
import json                                                                                                        
                                                                                                                   
# Convert the string into a JSON object                                                                            
result_json = json.loads(result)                                                                                   
                                                                                                                   
# Filter devices with high reliability and speed tiers                                                             
high_performance_devices = [device for device in result_json if device["reliability_tier"] == "high" and           
device["speed_tier"] == "high"]                                                                                    
                                                                                                                   
# If no devices with high reliability and speed tiers, consider mid reliability and low speed tiers                
if not high_performance_devices:                                                                                   
    mid_reliability_devices = [device for device in result_json if device["reliability_tier"] == "mid"]            
                                                                                                                   
# Select the cheapest device from the filtered list                                                                
if high_performance_devices:                                                                                       
    best_device = min(high_performance_devices, key=lambda x: x["price_rub"])                                      
else:                                                                                                              
    best_device = min(mid_reliability_devices, key=lambda x: x["price_rub"])                                       
                                                                                                                   
# Construct the final JSON                                                                                         
final_json = {                                                                                                     
    "normalized_name": best_device["normalized_name"],                                                             
    "capacity_gb": best_device["capacity_gb"],                                                                     
    "price_rub": best_device["price_rub"],                                                                         
    "source_url": best_device["source_url"]                                                                        
}                                                                                                                  
                                                                                                                   
final_answer(final_json)                                                                                           
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import json                                                                                                      
                                                                                                                   
  # Convert the string into a JSON object                                                                          
  result_json = json.loads(result)                                                                                 
                                                                                                                   
  # Filter devices with high reliability and speed tiers                                                           
  high_performance_devices = [device for device in result_json if device["reliability_tier"] == "high" and         
  device["speed_tier"] == "high"]                                                                                  
                                                                                                                   
  # If no devices with high reliability and speed tiers, consider mid reliability and low speed tiers              
  if not high_performance_devices:                                                                                 
      mid_reliability_devices = [device for device in result_json if device["reliability_tier"] == "mid"]          
                                                                                                                   
  # Select the cheapest device from the filtered list                                                              
  if high_performance_devices:                                                                                     
      best_device = min(high_performance_devices, key=lambda x: x["price_rub"])                                    
  else:                                                                                                            
      best_device = min(mid_reliability_devices, key=lambda x: x["price_rub"])                                     
                                                                                                                   
  # Construct the final JSON                                                                                       
  final_json = {                                                                                                   
      "normalized_name": best_device["normalized_name"],                                                           
      "capacity_gb": best_device["capacity_gb"],                                                                   
      "price_rub": best_device["price_rub"],                                                                       
      "source_url": best_device["source_url"]                                                                      
  }                                                                                                                
                                                                                                                   
  final_answer(final_json)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'normalized_name': 'MSI SPATIUM M461', 'capacity_gb': 1000, 'price_rub': 12663, 'source_url': 
'https://www.wildberries.ru/catalog/174666786/detail.aspx'}

[Step 3: Duration 1.42 seconds| Input tokens: 9,349 | Output tokens: 742]

{'normalized_name': 'MSI SPATIUM M461', 'capacity_gb': 1000, 'price_rub': 12663, 'source_url': 'https://www.wildberries.ru/catalog/174666786/detail.aspx'}


In [12]:
agent_case = make_agent(model_ll, prompts['CASE'])
min_price = request['budget'] * ratio[5][0]
max_price = request['budget'] * ratio[5][1]
psu_ff = psu['form_factor']
mb_ff = cpu_mb['form_factor']
selected_gpu_length = gpu['length_mm']

temp = f"""
min_price = {min_price}, max_price={max_price}, motherboard_form_factor = {mb_ff}, psu_form_factor = {psu_ff}, selected_gpu_length = {selected_gpu_length}
"""
try:
    res = agent_case.run(temp)
    if isinstance(res, str):
        clean_res = res.replace("Final answer:", "").strip()
        try:
            case = ast.literal_eval(clean_res)
        except Exception:
            case = res
    else:
        case = res
    print(case)
except Exception as e:
    print(f"\n❌ Ошибка исполнения: {e}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ min_price = 2400.0, max_price=7200.0, motherboard_form_factor = mATX, psu_form_factor = ATX,                    │
│ selected_gpu_length = 280                                                                                       │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-4-Scout-17B-16E-Instruct ──────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will now construct the SQL query using the provided parameters and then execute it to get the desired   
results.                                                                                                           
                                                                                                                   
```python                                                                                                          
min_price = 2400.0                                                                                                 
max_price = 7200.0                                                                                                 
motherboard_form_factor = "mATX"                                                                                   
psu_form_factor = "ATX"                                                                                            
selected_gpu_length = 280                                                                                          
                                                                                                                   
sql_query = f"""                                                                                                   
SELECT c.normalized_name, c.max_cooler_height_mm, c.puffability_tier, p.price_rub, p.source_url                    
FROM cases AS c                                                                                                    
INNER JOIN component_prices AS p ON c.id = p.component_id                                                          
WHERE p.is_available = TRUE                                                                                        
  AND p.price_rub BETWEEN {int(min_price)} AND {int(max_price)}                                                    
  AND '{motherboard_form_factor}' = ANY(c.supported_form_factors_motherboard)                                      
  AND '{psu_form_factor}' = ANY(c.supported_form_factors_psu)                                                      
  AND c.max_gpu_lenght_mm >= {selected_gpu_length}                                                                 
ORDER BY c.puffability_tier DESC, p.price_rub ASC                                                                  
"""                                                                                                                
                                                                                                                   
result = query_database(sql_query)                                                                                 
print(result)                                                                                                      
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  min_price = 2400.0                                                                                               
  max_price = 7200.0                                                                                               
  motherboard_form_factor = "mATX"                                                                                 
  psu_form_factor = "ATX"                                                                                          
  selected_gpu_length = 280                                                                                        
                                                                                                                   
  sql_query = f"""                                                                                                 
  SELECT c.normalized_name, c.max_cooler_height_mm, c.puffability_tier, p.price_rub, p.source_url                  
  FROM cases AS c                                                                                                  
  INNER JOIN component_prices AS p ON c.id = p.component_id                                                        
  WHERE p.is_available = TRUE                                                                                      
    AND p.price_rub BETWEEN {int(min_price)} AND {int(max_price)}                                                  
    AND '{motherboard_form_factor}' = ANY(c.supported_form_factors_motherboard)                                    
    AND '{psu_form_factor}' = ANY(c.supported_form_factors_psu)                                                    
    AND c.max_gpu_lenght_mm >= {selected_gpu_length}                                                               
  ORDER BY c.puffability_tier DESC, p.price_rub ASC                                                                
  """                                                                                                              
                                                                                                                   
  result = query_database(sql_query)                                                                               
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[
  {
    "price_rub": 7003,
    "source_url": "https://www.wildberries.ru/catalog/756640637/detail.aspx",
    "normalized_name": "Cougar Airface Pure Pro [Airface Pure Pro Black]",
    "puffability_tier": "mid",
    "max_cooler_height_mm": 175
  },
  {
    "price_rub": 6151,
    "source_url": "https://www.wildberries.ru/catalog/691742477/detail.aspx",
    "normalized_name": "ARDOR GAMING Rare Minicase MS3",
    "puffability_tier": "low",
    "max_cooler_height_mm": 160
  },
  {
    "price_rub": 3577,
    "source_url": "https://www.wildberries.ru/catalog/283340020/detail.aspx",
    "normalized_name": "ZALMAN N4 Rev.1",
    "puffability_tier": "high",
    "max_cooler_height_mm": 163
  },
  {
    "price_rub": 6247,
    "source_url": "https://www.wildberries.ru/catalog/236207586/detail.aspx",
    "normalized_name": "MONTECH X2 MESH [X2 (B)]",
    "puffability_tier": "high",
    "max_cooler_height_mm": 165
  }
]

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I have received the results from the database query. Now, I need to apply the selection algorithm to      
identify the best case based on the provided criteria.                                                             
                                                                                                                   
1. Identify the highest available puffability_tier.                                                                
2. Among cases with that highest tier, pick the one with the lowest price_rub.                                     
3. If prices are equal, pick the one that was first in the database response.                                      
                                                                                                                   
```python                                                                                                          
# Results from the database query                                                                                  
cases = [                                                                                                          
  {                                                                                                                
    "price_rub": 7003,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/756640637/detail.aspx",                                      
    "normalized_name": "Cougar Airface Pure Pro [Airface Pure Pro Black]",                                         
    "puffability_tier": "mid",                                                                                     
    "max_cooler_height_mm": 175                                                                                    
  },                                                                                                               
  {                                                                                                                
    "price_rub": 6151,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/691742477/detail.aspx",                                      
    "normalized_name": "ARDOR GAMING Rare Minicase MS3",                                                           
    "puffability_tier": "low",                                                                                     
    "max_cooler_height_mm": 160                                                                                    
  },                                                                                                               
  {                                                                                                                
    "price_rub": 3577,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/283340020/detail.aspx",                                      
    "normalized_name": "ZALMAN N4 Rev.1",                                                                          
    "puffability_tier": "high",                                                                                    
    "max_cooler_height_mm": 163                                                                                    
  },                                                                                                               
  {                                                                                                                
    "price_rub": 6247,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/23

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Results from the database query                                                                                
  cases = [                                                                                                        
    {                                                                                                              
      "price_rub": 7003,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/756640637/detail.aspx",                                    
      "normalized_name": "Cougar Airface Pure Pro [Airface Pure Pro Black]",                                       
      "puffability_tier": "mid",                                                                                   
      "max_cooler_height_mm": 175                                                                                  
    },                                                                                                             
    {                                                                                                              
      "price_rub": 6151,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/691742477/detail.aspx",                                    
      "normalized_name": "ARDOR GAMING Rare Minicase MS3",                                                         
      "puffability_tier": "low",                                                                                   
      "max_cooler_height_mm": 160                                                                                  
    },                                                                                                             
    {                                                                                                              
      "price_rub": 3577,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/283340020/detail.aspx",                                    
      "normalized_name": "ZALMAN N4 Rev.1",                                                                        
      "puffability_tier": "high",                                                                                  
      "max_cooler_height_mm": 163                                                                                  
    },                                                                                                             
    {                                                                                                              
      "price_rub": 6247,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/236207586/detail.aspx",                                    
      "normalized_name": "MONTECH X2 MESH [X2 (B)]",                                                               
      "puffability_tier": "high",                                                                                  
      "max_cooler_height_mm": 165                                                                                  
    }                                                                                                              
  ]                                                                                                                
                                                                                                                   
  # Step 1: Identify the highest available puffability_tier                                                        
  tiers = [case["puffability_tier"] for case in cases]   

Final answer: {'normalized_name': 'ZALMAN N4 Rev.1', 'puffability_tier': 'high', 'max_cooler_height_mm': 163, 
'price_rub': 3577, 'source_url': 'https://www.wildberries.ru/catalog/283340020/detail.aspx'}

[Step 2: Duration 2.39 seconds| Input tokens: 5,901 | Output tokens: 823]

{'normalized_name': 'ZALMAN N4 Rev.1', 'puffability_tier': 'high', 'max_cooler_height_mm': 163, 'price_rub': 3577, 'source_url': 'https://www.wildberries.ru/catalog/283340020/detail.aspx'}


In [13]:
agent_cooler = make_agent(model_ll, prompts['COOLER'])
min_price = request['budget'] * ratio[6][0]
max_price = request['budget'] * ratio[6][1]
max_height  = case['max_cooler_height_mm']
tdp = cpu_mb['tdp']
socket = cpu_mb['socket']


temp = f"""
min_price = {min_price}, max_price={max_price}, selected_socket = {socket}, selected_cpu_tdp = {tdp}, selected_case_max_cooler_height = {max_height}
"""
try:
    res = agent_cooler.run(temp)
    if isinstance(res, str):
        clean_res = res.replace("Final answer:", "").strip()
        try:
            cooler = ast.literal_eval(clean_res)
        except Exception:
            cooler = res
    else:
        cooler = res
    print(cooler)
except Exception as e:
    print(f"\n❌ Ошибка исполнения: {e}")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ min_price = 1200.0, max_price=4800.0, selected_socket = AM4, selected_cpu_tdp = 65,                             │
│ selected_case_max_cooler_height = 163                                                                           │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-4-Scout-17B-16E-Instruct ──────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: First, I need to calculate the required TDP, which is 1.5 times the selected CPU TDP. Then, I will        
construct the SQL query to find the suitable coolers.                                                              
                                                                                                                   
```python                                                                                                          
# Given parameters                                                                                                 
min_price = 1200.0                                                                                                 
max_price = 4800.0                                                                                                 
selected_socket = "AM4"                                                                                            
selected_cpu_tdp = 65                                                                                              
selected_case_max_cooler_height = 163                                                                              
                                                                                                                   
# Calculate required TDP                                                                                           
required_tdp = selected_cpu_tdp * 1.5                                                                              
print(f"Required TDP: {required_tdp}")                                                                             
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Given parameters                                                                                               
  min_price = 1200.0                                                                                               
  max_price = 4800.0                                                                                               
  selected_socket = "AM4"                                                                                          
  selected_cpu_tdp = 65                                                                                            
  selected_case_max_cooler_height = 163                                                                            
                                                                                                                   
  # Calculate required TDP                                                                                         
  required_tdp = selected_cpu_tdp * 1.5                                                                            
  print(f"Required TDP: {required_tdp}")                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Required TDP: 97.5

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I have received the list of suitable coolers. Now, I need to pick the one with the highest TDP and the    
lowest price. If there are multiple coolers with the same TDP, I will choose the one with the lowest price.        
                                                                                                                   
```python                                                                                                          
# Given data                                                                                                       
coolers = [                                                                                                        
  {                                                                                                                
    "tdp": 300,                                                                                                    
    "height_mm": "163",                                                                                            
    "price_rub": 4164,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/218486297/detail.aspx?targetUrl=MI",                         
    "normalized_name": "ID-COOLING FROZN A720"                                                                     
  },                                                                                                               
  {                                                                                                                
    "tdp": 260,                                                                                                    
    "height_mm": "157",                                                                                            
    "price_rub": 2819,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/789126375/detail.aspx?targetUrl=MI",                         
    "normalized_name": "ID-COOLING FROZN A620 PRO SE"                                                              
  },                                                                                                               
  {                                                                                                                
    "tdp": 250,                                                                                                    
    "height_mm": "155",                                                                                            
    "price_rub": 2573,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/214720453/detail.aspx?targetUrl=MI",                         
    "normalized_name": "ID-COOLING FROZN A610"                                                                     
  },                                                                                                               
  {                                                                                                                
    "tdp": 220,                                                                                                    
    "height_mm": "150",                                                                                            
    "price_rub": 1407,                                                                                             
    "source_url": "https://www.wildberries.ru/catalog/564367369/detail.aspx?targetUrl=MI",                         
    "normalized_name": "Deepcool AG400"                                                                            
  },                                                    

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Given data                                                                                                     
  coolers = [                                                                                                      
    {                                                                                                              
      "tdp": 300,                                                                                                  
      "height_mm": "163",                                                                                          
      "price_rub": 4164,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/218486297/detail.aspx?targetUrl=MI",                       
      "normalized_name": "ID-COOLING FROZN A720"                                                                   
    },                                                                                                             
    {                                                                                                              
      "tdp": 260,                                                                                                  
      "height_mm": "157",                                                                                          
      "price_rub": 2819,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/789126375/detail.aspx?targetUrl=MI",                       
      "normalized_name": "ID-COOLING FROZN A620 PRO SE"                                                            
    },                                                                                                             
    {                                                                                                              
      "tdp": 250,                                                                                                  
      "height_mm": "155",                                                                                          
      "price_rub": 2573,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/214720453/detail.aspx?targetUrl=MI",                       
      "normalized_name": "ID-COOLING FROZN A610"                                                                   
    },                                                                                                             
    {                                                                                                              
      "tdp": 220,                                                                                                  
      "height_mm": "150",                                                                                          
      "price_rub": 1407,                                                                                           
      "source_url": "https://www.wildberries.ru/catalog/564367369/detail.aspx?targetUrl=MI",                       
      "normalized_name": "Deepcool AG400"                                                                          
    },                                                                                                             
    {                                                                                                              
      "tdp": 220,                                                                                                  
      "height_mm": "151",                                                                                          
      "price_rub": 1971,                                 

Final answer: {'normalized_name': 'ID-COOLING FROZN A720', 'tdp': 300, 'height_mm': 163, 'price_rub': 4164, 
'source_url': 'https://www.wildberries.ru/catalog/218486297/detail.aspx?targetUrl=MI'}

[Step 4: Duration 2.42 seconds| Input tokens: 12,724 | Output tokens: 1,135]

{'normalized_name': 'ID-COOLING FROZN A720', 'tdp': 300, 'height_mm': 163, 'price_rub': 4164, 'source_url': 'https://www.wildberries.ru/catalog/218486297/detail.aspx?targetUrl=MI'}


In [14]:
print('GPU: ', gpu)
print('CPU+MB: ', cpu_mb)
print('RAM: ', ram)
print('STORAGE: ', disk)
print('PSU: ', psu)
print('CASE: ', case)
print('COOLER: ', cooler)

GPU:  {'normalized_name': 'Sapphire AMD Radeon RX 9070 PULSE', 'price_rub': 57544, 'tdp': 220, 'length_mm': 280, 'power_connectors': '16', 'source_url': 'https://www.wildberries.ru/catalog/834513944/detail.aspx'}
CPU+MB:  {'cpu_name': 'AMD Ryzen 5 5600', 'motherboard_name': 'ASRock B550M-HDV', 'cpu_and_mb_price': 17228, 'tdp': 65, 'socket': 'AM4', 'form_factor': 'mATX', 'ram_type': 'DDR4', 'num_ram_slots': 2, 'cpu_power_pins': 4, 'required_cpu_power_pins': 4, 'motherboard_url': 'https://www.wildberries.ru/catalog/732500105/detail.aspx', 'cpu_url': 'https://www.wildberries.ru/catalog/377609179/detail.aspx'}
RAM:  {'normalized_name': 'Kingston Fury DDR4 3200MHz 16Gb', 'total_capacity_gb': 16, 'speed_mhz': 3200, 'price_rub': 12700, 'source_url': 'https://www.wildberries.ru/catalog/150972149/detail.aspx'}
STORAGE:  {'normalized_name': 'MSI SPATIUM M461', 'capacity_gb': 1000, 'price_rub': 12663, 'source_url': 'https://www.wildberries.ru/catalog/174666786/detail.aspx'}
PSU:  {'normalized_nam